# 04 — Clean-Room Photo-to-Lineart (#9, ~6 MB) & Auto-Colorize (#12, ~7 MB)
100% commercial-clean (zero OpenSketch/Contour/ImageNet taint):
- **#9 `lineart_clean.onnx` (MIT):** Compact ResNet generator trained on Infinite `e1_200` G-buffer normal+depth edges + PD12M CC0 structural edges.
- **#12 `colorize_clean.onnx` (Apache-2.0):** Self-supervised `MobileNetV4-Small` CIE-Lab colorizer (`L -> ab`) trained on PD12M CC0 colour photos.
Both export static NCHW ONNX straight to `/content/drive/MyDrive/infinite_ml/`.


In [ ]:
!pip -q uninstall -y torchaudio
!pip -q install -U torch torchvision timm onnx onnxscript onnxslim onnxruntime datasets pillow numpy tqdm
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/out', exist_ok=True)
!unzip -q -o /content/drive/MyDrive/infinite_ml/e1_200.zip -d /content/


In [ ]:
# Shared clean data: stream 300 CC0 photos from Spawning/PD12M
import os, glob, random, shutil, hashlib, urllib.request
from io import BytesIO
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, onnx, timm
from PIL import Image, ImageFilter
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

pd_dir = '/content/pd12m_300'
os.makedirs(pd_dir, exist_ok=True)
if len(glob.glob(f'{pd_dir}/*.jpg')) < 260:
    ds_stream = load_dataset('Spawning/PD12M', split='train', streaming=True)
    saved = 0
    for row in ds_stream:
        if saved >= 300: break
        url = row.get('url') or row.get('image_url') or ''
        if not url.startswith('http'): continue
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'infinite-ml'})
            with urllib.request.urlopen(req, timeout=5) as r:
                im = Image.open(BytesIO(r.read())).convert('RGB')
            if min(im.size) < 256: continue
            im.resize((512, 512), Image.BILINEAR).save(f'{pd_dir}/pd_{saved:04d}.jpg', quality=92)
            saved += 1
        except Exception:
            continue
    print('Fetched PD12M CC0 photos:', saved)
pd_imgs = sorted(glob.glob(f'{pd_dir}/*.jpg'))
render_dirs = sorted(glob.glob('/content/e1_200/scene_*'))
print(f'Corpus ready: {len(render_dirs)} Infinite renders + {len(pd_imgs)} PD12M CC0 photos')


In [ ]:
# ==================== #9 CLEAN LINE-ART (Photo -> Ink Sketch) ====================
def sobel_mag(arr):
    # arr: [H, W, C] float32 in [0, 1]
    gx = np.abs(arr[:, 2:, :] - arr[:, :-2, :]).mean(axis=2)
    gy = np.abs(arr[2:, :, :] - arr[:-2, :, :]).mean(axis=2)
    m = np.zeros(arr.shape[:2], np.float32)
    m[1:-1, 1:-1] = np.sqrt(gx[1:-1, :]**2 + gy[:, 1:-1]**2)
    return m

def make_render_lineart(d, size=512):
    nrm = np.asarray(Image.open(os.path.join(d, 'normal.png')).convert('RGB').resize((size, size)), np.float32) / 255.0
    dep = np.asarray(Image.open(os.path.join(d, 'depth.png')).convert('L').resize((size, size)), np.float32)[:, :, None] / 255.0
    alp = np.asarray(Image.open(os.path.join(d, 'alpha.png')).convert('L').resize((size, size)), np.float32)[:, :, None] / 255.0
    edge = np.clip(sobel_mag(nrm) * 2.8 + sobel_mag(dep) * 4.0 + sobel_mag(alp) * 3.0, 0.0, 1.0)
    return 1.0 - edge  # white paper (1.0), black ink lines (0.0)

def make_photo_lineart(p, size=512):
    im = Image.open(p).convert('L').resize((size, size), Image.BILINEAR)
    g1 = np.asarray(im.filter(ImageFilter.GaussianBlur(0.8)), np.float32) / 255.0
    g2 = np.asarray(im.filter(ImageFilter.GaussianBlur(2.2)), np.float32) / 255.0
    dog = np.clip((g2 - g1) * 8.0, 0.0, 1.0)
    return 1.0 - dog

class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c, c, 3, padding=1), nn.InstanceNorm2d(c), nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1), nn.InstanceNorm2d(c),
        )
    def forward(self, x): return x + self.net(x)

class CleanLineArtNet(nn.Module):
    def __init__(self, c=48, n_blocks=6):
        super().__init__()
        self.down = nn.Sequential(
            nn.Conv2d(3, c, 7, padding=3), nn.InstanceNorm2d(c), nn.ReLU(inplace=True),
            nn.Conv2d(c, c*2, 3, stride=2, padding=1), nn.InstanceNorm2d(c*2), nn.ReLU(inplace=True),
            nn.Conv2d(c*2, c*2, 3, stride=2, padding=1), nn.InstanceNorm2d(c*2), nn.ReLU(inplace=True),
        )
        self.mid = nn.Sequential(*[ResBlock(c*2) for _ in range(n_blocks)])
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(c*2, c, 3, padding=1), nn.InstanceNorm2d(c), nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(c, c//2, 3, padding=1), nn.InstanceNorm2d(c//2), nn.ReLU(inplace=True),
            nn.Conv2d(c//2, 1, 7, padding=3), nn.Sigmoid(),
        )
    def forward(self, x): return self.up(self.mid(self.down(x)))

class LineArtDS(Dataset):
    def __init__(self, rdirs, pimgs, size=512):
        self.rdirs, self.pimgs, self.size = rdirs, pimgs, size
    def __len__(self): return len(self.rdirs) + len(self.pimgs)
    def __getitem__(self, i):
        if i < len(self.rdirs):
            d = self.rdirs[i]
            rgb = np.asarray(Image.open(os.path.join(d, 'composite.png')).convert('RGB').resize((self.size, self.size)), np.float32) / 255.0
            ink = make_render_lineart(d, self.size)
        else:
            p = self.pimgs[i - len(self.rdirs)]
            rgb = np.asarray(Image.open(p).convert('RGB').resize((self.size, self.size)), np.float32) / 255.0
            ink = make_photo_lineart(p, self.size)
        return torch.from_numpy(rgb).permute(2,0,1), torch.from_numpy(ink)[None]

dl_la = DataLoader(LineArtDS(render_dirs, pd_imgs), batch_size=8, shuffle=True, num_workers=2)
la_net = CleanLineArtNet(c=48, n_blocks=6).cuda()
opt = torch.optim.AdamW(la_net.parameters(), lr=5e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)
for ep in range(20):
    tot = 0.0
    for rgb, ink in dl_la:
        rgb, ink = rgb.cuda(), ink.cuda()
        pred = la_net(rgb)
        loss = F.l1_loss(pred, ink) + 0.5 * F.binary_cross_entropy(pred.clamp(1e-5, 1 - 1e-5), ink)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()
    sched.step()
    if (ep + 1) in (1, 5, 10, 15, 20):
        print(f"LineArt Epoch {ep+1}/20 loss={tot/len(dl_la):.4f}")

out_la = '/content/out/lineart_clean.onnx'
torch.onnx.export(la_net.eval().cpu(), torch.randn(1,3,512,512), out_la, opset_version=17, dynamo=False,
                  input_names=['image'], output_names=['lineart'])
onnx.checker.check_model(out_la)
d_la = open(out_la, 'rb').read()
shutil.copy2(out_la, '/content/drive/MyDrive/infinite_ml/lineart_clean.onnx')
print(f"LineArt Exported: {len(d_la)/1e6:.2f} MB sha256={hashlib.sha256(d_la).hexdigest()}")


In [ ]:
# ==================== #12 CLEAN COLORIZE (Self-Supervised YCbCr on PD12M CC0) ====================
# Uses BT.601 YCbCr (exact reversible math in C++ with zero external dependencies):
#   Input: Y in [0, 1] (1x1x256x256) -> Output: Cb, Cr in [-0.5, 0.5] (1x2x256x256)
def rgb_to_ycbcr(rgb):
    r, g, b = rgb[:,:,0], rgb[:,:,1], rgb[:,:,2]
    y  =  0.299 * r + 0.587 * g + 0.114 * b
    cb = -0.168736 * r - 0.331264 * g + 0.5 * b
    cr =  0.5 * r - 0.418688 * g - 0.081312 * b
    return y.astype(np.float32), np.stack([cb, cr], axis=0).astype(np.float32)

# Keep only genuinely colourful PD12M photos (filter out B&W / sepia archival images)
color_imgs = []
for p in pd_imgs:
    arr = np.asarray(Image.open(p).convert('RGB').resize((128, 128)), np.float32) / 255.0
    _, cbcr = rgb_to_ycbcr(arr)
    if float(np.std(cbcr)) > 0.035:
        color_imgs.append(p)
print(f'Colourful PD12M CC0 photos kept: {len(color_imgs)} / {len(pd_imgs)}')

class CleanColorizeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = timm.create_model('mobilenetv4_conv_small.e2400_r224_in1k', pretrained=True, features_only=True, out_indices=(1,2,3,4))
        chs = self.enc.feature_info.channels()
        self.lat = nn.ModuleList([nn.Conv2d(c, 64, 1) for c in chs])
        self.head = nn.Sequential(
            nn.Conv2d(64*4, 96, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(96, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 2, 1), nn.Tanh()
        )
    def forward(self, luma):
        H, W = luma.shape[2], luma.shape[3]
        x3 = luma.repeat(1, 3, 1, 1)
        feats = [F.interpolate(l(f), size=(H//4, W//4), mode='bilinear', align_corners=False)
                 for l, f in zip(self.lat, self.enc(x3))]
        return 0.5 * F.interpolate(self.head(torch.cat(feats, dim=1)), size=(H, W), mode='bilinear', align_corners=False)

class ColorDS(Dataset):
    def __init__(self, paths, size=256):
        self.paths, self.size = paths, size
    def __len__(self): return len(self.paths) * 2
    def __getitem__(self, i):
        im = Image.open(self.paths[i % len(self.paths)]).convert('RGB').resize((self.size, self.size), Image.BILINEAR)
        if random.random() < 0.5: im = im.transpose(Image.FLIP_LEFT_RIGHT)
        y, cbcr = rgb_to_ycbcr(np.asarray(im, np.float32) / 255.0)
        return torch.from_numpy(y)[None], torch.from_numpy(cbcr)

dl_col = DataLoader(ColorDS(color_imgs, 256), batch_size=16, shuffle=True, num_workers=2)
col_net = CleanColorizeNet().cuda()
opt = torch.optim.AdamW(col_net.parameters(), lr=5e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=25)
for ep in range(25):
    tot = 0.0
    for y, cbcr in dl_col:
        y, cbcr = y.cuda(), cbcr.cuda()
        pred = col_net(y)
        # Weight vibrant pixels slightly higher so colour doesn't wash out to sepia
        w = 1.0 + 3.0 * cbcr.abs().mean(dim=1, keepdim=True)
        loss = (F.smooth_l1_loss(pred, cbcr, reduction='none') * w).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()
    sched.step()
    if (ep + 1) in (1, 5, 10, 15, 20, 25):
        print(f"Colorize Epoch {ep+1}/25 loss={tot/len(dl_col):.4f}")

out_col = '/content/out/colorize_clean.onnx'
torch.onnx.export(col_net.eval().cpu(), torch.randn(1,1,256,256), out_col, opset_version=17, dynamo=False,
                  input_names=['luma'], output_names=['cbcr'])
onnx.checker.check_model(out_col)
d_col = open(out_col, 'rb').read()
shutil.copy2(out_col, '/content/drive/MyDrive/infinite_ml/colorize_clean.onnx')
print(f"Colorize Exported: {len(d_col)/1e6:.2f} MB sha256={hashlib.sha256(d_col).hexdigest()}")
